# Disentangling Representation Collapse in Ticktack Baseline (Han et al., 2025)

This notebook executes a rigorous, independent empirical study to determine the exact root cause of representation collapse and verify letter bias debiasing on TempUN Small (2,461 train, 180 eval):

### 4-Way Control Configurations:
- **`Base` (Zero-shot)**: Frozen `Qwen/Qwen2.5-0.5B-Instruct` on raw prompt `Question: ...\nAnswer:`.
- **`C1 (Raw Broadcast)`**: TE broadcast across all tokens with $\sigma = 0.5$ (Literal paper formula Eq. 1-5, Eq. 8).
- **`C2 (Raw Date-token only)`**: TE injected only at date token positions with $\sigma = 0.5$ (Prevents RMSNorm from washing out token identities).
- **`C3 (Raw with $\sigma = 0$)`**: TE broadcast across tokens, but $\sigma = 0.0$ (Pure NTP, no intra-class contrastive loss).
- **`C4 (PT + Intra loss)`**: No TE injected, but trained with NTP + $\sigma \times$ intra-loss ($\sigma = 0.5$, tests intra-loss alone).

### Evaluation & Debiasing Protocol:
- **Academic Choice Log-Likelihood Scoring** (Invalid Rate = 0.0%).
- **4-fold Option Cyclic Permutation Rotation** (Rot 0: ABCD, Rot 1: BCDA, Rot 2: CDAB, Rot 3: DABC) for MCQA.
- **Bidirectional Ordering** ('Yes or No', 'No or Yes') for Yes/No questions.
- Measures **Canonical Accuracy**, **Debiased Mean Accuracy**, and **Consistency (Strict Debiased Accuracy)**.

In [ ]:
import os, sys, time, math, gzip, base64, json, random, gc, re
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import stats

print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    device = torch.device('cuda:0')
    print('GPU Device Name:', torch.cuda.get_device_name(0))
    print('Total VRAM:', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), 'GB')
else:
    device = torch.device('cpu')
    print('WARNING: GPU not found, running on CPU!')


In [ ]:
TRAIN_B64 = '...'
EVAL_B64 = 'H4sIAAAAAAAC/+19W3MbR5L1+/4KhF8+O8LE1P3ifZiQNB7ba8t2SFp7PS8KkIQorEGAAxDS0Bv7379Tt0ZlEyCbRFNaU+0I20QDqOrGyXtmZf7PZ9N/Tc4v5tPXs9PPvhp9dn7yz6+uppPV/OrozXJ1Prk8mr6bzI9OJ5eT9fTyq1fT84v//PH1y/PJfP6X8M5mcjlbLl5frJbnF5frv+z/6l/Op6evV9M5Pv9u+voYl05fY7P8zdfpi/nVX7hXdnyyfvfV02f/9Y8jrvwROwrXvlot33/Fmf3sy9Fns8XF5jLc8neLUXjvy9HPq9m7yeV09Gz2bjYf/TA7nq4uZ9P16LvF6fRfo9li9PNydbk5m8xH7yfr0dvZ2dvpanT5dhK+r8VfR08W6/e4gvd+w7eWq9GPy7DR5WR1No074XK4EO4VL9Omn51MFidvZ69Prk7Cjxi2wntCxS+uf88/67OfwxeXq9nZbDGZv8Z3lovZCf7Kay028zk+MH3zZnoSf6Bqj//9t//5M4CkNQGJ83uC9HKyHL1ank9Hk8Vp+PTiZHYxHRB7CMQMRUzcG7HN6Wz0ZDU5nk0GoB4AKCspUPeVfy8vJvjvgFDvCFleA2T1PfH5ZjNbTCdHT2fr9WRzX5zS6wGmXTARPrLmnjB9O5ldzgZ4+ofHEXjuK+W+XS7ORt+H/wwQ9QjR3375+QXQsTVGYo+kW7+drKaj5ZvRyXJxuZodb8KdjS6Xo7P58jja36vzGQDCJ17OpqvVZPTDdLkYrLwHQExwIvSEPRyx99PT6WBGPARWitgR8nDu+m16PkDVM1TfPP2vr8FW1N3VajdWl2+no4vlxWYe72kExlnPLq+iMX61GtylvqH55cVPR5wRiafYDchMV7Pl6Wg+ezMdTf91gSfGT3g1mlyOjmery7cBJpgV0EzrAar9UE0n66vXeHFXt9YoghN3BKhf385O3o6WF5FvZmsIu9UKzznCHUTs/rmZrsN7X43u4ADPl+8mvycsvxr9FNfGH59Pvhixseajz4/DH0qOPj8Jfxg/+vz0ixEfC0MxPb4non97ejOiea0/IaJJfzEiFAUzhyJ6F7NkuQHHPjkHS5/sAtiPuYsAczMWPCLsx0wmhNXYtiA+GSDeBbEhMUNjPyDAX/8TT4cfFldToGQnF7NrXCwixOBvRxGeDAjvQtgTHvYfkoWbwEoLVT5mJqPKbEbV6QyrUINs7mKwMpJEM1IdCuwNpm2JM7dwNGKsE5CGjb2PSOogeiOU2o2NHoRwJ+eDSmHFHw7LV5PZ+8kuMAUD78mIptBm7FWEUzA5VknicuvHsqVVTwdAdwEqBbGE3QPy5tPVZjHdJWTBfSKiiT+S6nTjZP8CRjWozg4w0vCN8uLhYHw2hTKF9nzyJpi7i9GL6cXmeD472cWoY5eELqSviMDKMU8iV4zFwJ83A/vixbfPjxThT+N6AXaxOYeDGuye89l8BgyuQjxivVwspvNRCX0EQ6lFBQH95fpyMnqx29FJfmu0lnSxgbnaYy0NGnYn4JYC7uVHBvyb1XR6Mt3J3EoX5tZJA4+lzmELJwfu7gC2UxRs95HB/u4Eb6P4aRdru8zZIjM2y3zNBrbugjTJXlrGPjbSq52mdYhVZJwtz0B7mZE2/NOyxRA+OH19uZouTu8YXmY6RTDcrvg/qMDoL0fv3yLWD6gDNDG2v74crUIIOcQrEOQ/m0b0bw0sP5/MJ++jVf3X0U+by/D1sGSipb9e5+ISznBFQXtXGFn3I7NfPf1zJA/uDa+UN5UtAl8t+8P3p5PpZJHyBp0A5iUKyVUR1CUM6XsKND92gL1I7Kv2sS/rD97vp4ur7uA2iSJj2+AyPnBvJ3BzdFnv490eZfMPk8t3nVmXB4vquGCa8gem+E49ZQEfPbgRW832YGt7lMtPNuvL1V3ksi7ZId9Gty+76hGjm0oAmUqKV+7jXdEZ3zvkhoqp3AnmJgmoi/kcrkSYpR1g7gSzTmws3AeF+cfZ2fQuDM0EtaT5FmnnB6Q7IW1zuaH4oEj//Bbe8sUFEvnr7pZX0c2Zq3UR3pwPqrkT1i55TfLDCu8Xm/X6LkzduMc6Ay0L0E4NTN2pFEuUqroPCvSrTXi6zkCrwtCmxEG03h3QHByp3SlHLk2yxhw7HOj96cbnkxWoYD4ffbcOoemuEtuMjUr1HmOWAiG6VHuYsdVDIKRLPbNM9raSPQF8e83z0838bNLdCpM5KyFLTY8q+UY5/sSimaFq/T0eYvn+jtFMntg42axVRmILa8lG5PxEgfPNcj5fvg/iF+mJ1dVXo79j/fjcQUJz72WHuujnOC23nO+si25SHRDS/55ehchbcKVdfu3ja6fL+55/CTu8el+E1yI7ZdWCXJIVhaIrKlevGLLZuloRO9okU6oVnSMrVivE19LRBS29RStTkqVakNFbZJIuKDy9RWkMvUXm+mGAXx87A7isx4TtmwOc68ABrzaL2fo2BhDOZnS1ifQjykubKL6Ar114LVV57eLXrb3GAYY7sqIxvloypP/IikwKsqLjvs0B0iuyomearKisJ0sKpsiS3rZZgLfuUXjRWlFUK+IZtCYrGukHHujCAynhoUTvKkB3YIAn8+OS4drPAKyQY8idhlgaFdhMSaICtKTyFZ0r2gzAmCBLsmaLvCQXRMAaTwWsFa7NAZauaJ0iK3ovyE1KZuhNOt7mACXpcwtPl3SKqj7FFdUCfYUnHjsHpJCjMn1zgO1iBFW5hH0cAJL2jThl6XUjTnV8LXwlLGFSFMqwySiSrs0BltMVpXdkRSYUWVFxsqLwss0AZvsNlpSKqVbEjkoTge58tSIYRFxjAIhwsqLfKkKdlIgl96g9uUcpeD+u3iOm/xy1YSk8p/rmgOAk3iWOszxb3qwKQAKwRlhjsquo+51guK4aglM+2dnaIJ9gHCEiZySiFrxZQfNkYylrmZfXNIU0sA0Na/gvbcid4lKohqLThlxprzhvdEvaUGjNhfNGkA0dk95Yr6+xERPaMXzFkA2ZMDj7bRrXJW2IqwZ61lqyIROWW6dbT4irGn6C8G0uc0wpqZSgG3oFDYWfj5MNjTLaKdH4N2lDLjz+Qb8osiH430hptBiYsBsT5hi5+dhM2MUngZOhlVaSEg3IDnWspuG4QDTBfQDXysbCSjQjNVOaVa5N4gqOQ0ZG6WvqCm49+jEaRbazSnoOAqNMoQ3WZYySKGwhbhlvfIS0nZeBQKUT15hQ46QaVKClTKgN9gBnUSaEnFFWtXgQ9+uc5vTxcNVAcuTkf+3iSayCk0uC7IftEFDnju6Hr4M3PVWNEhEXLYylQgadPSxCtWxgwW5nUHMw4GNz4LcIpR6hA2VoR3my3ISPp2zlPn7ECUclZOMGBerRHjaatzVr4FPGg5FqMw3nIKXSWtR0oxFRhZZsqUJtx8xtv5z34Eo1LBz2wHcZ/DMj6z00bD1wtan3UOOgPn2L8/D1YL2RB8GKBgGtehOB+AKjFiyuWYF/agbX4XRC4FjKbtgbKtY6+iQKwkDTJzGWOVVbD/gNLGeNm5X3CNrWDaZmt8SR0LlzDX8IJtufSvphdvl2c1uoQQVPH9qq1moKKSMYQhVl4Ipm3NXUh0tC+63PHyhDIUGhECpr8ZEKjokTmuwBXaabGFzcw4IHfc2p+J4O5lvNqQrcK2AqtgLSWA5bOCvpFpq5movCptAtNafiEqJp23uLe4SIIgzVFhPhsvNyy+hxDxwCMsQmDlek5LWuwpM5MBGrjUWlEc4LNvXARN0O1WcmMvrDMlFudvJmT6+TbcACZ3cFPJOaODgCWpzVfMQtCuVEbadxyHqtlKk9F1yDhWa8omyEywYMwmo24hrbSsuJ+WngHVqpa+tThJMusLh8vY3HzWicaaesxEMgGm6QoE8i4eS52tvkISwvuK+5iQeuFpV3xuM1D57zrYwPdyENZYl9i6fhXDcWXdwFcRhthK35Cb8irMFtBijuAl3NYcCagZ+68ZPL/GQ+Bj+93JzuPI+1lbVjYZkWdQgCJhq0kvO17YVrwXeytcWiEO+AachrUkfrC8QMXDsxCnWFo2meW7JNUCNSa7oNQi1a1w6PDPECwxo1GbYJeXmrGLct5YSAA9wgVj+NQgwfB6cUUU9IXCHwTWgd34WaZY7oJ9h4+BTXLf0EQxI2nlRkG22lIjEU7KzxzJz+ZnC6OOIt9DcTyggth4h6J44SInOU/7Ac9QolBTeqJouwMRIrtfFvQzYGRF7LcwPd5AlV2JDD0cLWVGFNJBVJGcnAibFGunoPo9FDCP8ougfClrUZaeC0wdgSNX2HG0YkrcVFuBqoloQlsYdFHtaQPRxUnamVn4VlqYL6os9hgtlLWcigvYoTNOiCO9RaWuKOWYk7JOIAvxXYWZCQp4UpyEU7FDEw0J4jw4l97IMZeAccH44Zq8n81pwVZ4jLSwS6lQmBNhqxlw4C1ckQHIe/TwOGCplMvAvJrGC7aRLoguKCCYRFuaxifCmaB5GNHUNkXtoWT4JdPA+WJQLdQatwYmmF7CtD4UDwZ3DbNKrvNO4DvBZi+MpqYq/CpDU27GmQpFWsDj6AbWWI3iFNAM/QtyomsKVCwA6ftBJhP0GERXhTIyOBGAf+hWYlNmDIA1v40QzPgdokY+kNIQiIOD9SBUiIaGJ6hjoOz4yGAQrJ0WJ2vAtfVQIpFkKJshX6hIKF04poK7OCC0l/IQ3XzcDIgPXBpdU0Mgl5ZJBhsDBvlbGeIIbwLexwD02NsC//9Mo1uneadaoq2EC/S3cUru2sXUo1MKpD6vkfk/Pds1Ocv0eH2bDlboh4Xw1mnfo/j1CKcll7JG0FkVO7IdpvyNwwNGpA5zB0pKHgGHZXcJ6BcZanA+s8ADi2Dc6dOefvs/+eDcD0Ckw4DwDTHWcCKmSYuAGZ26v9X6JZ5+XoF6S50FMuTl0LX0THqcXktJzOGyDsC8LoVEiNHp4Vgv4G3jqkTdwNTQLvg2hrMM4AaAOol6GHZwOoYPxBAP2PycVkMfBjn/D99tt3P2AUAZRdzZDK78ZvMbncBI46w6cgPeMRq8+Dxx/Oxq1hzAfxOTmbfjEKrcJwh4ACuJVT7bOB7XrFDVENipsWveL2cro6HmDrHzbRhk33ClvuWT3A1hts/3jx3bfgNn4E5BrYtNuN2uQM5sZmDugwxHeFOQ2A5G2wPFcJnlbnh0GX9YqSlkeVJSL9/TB6tjxfrpYDPj00IM3gpP4b7ZCHPKizcPhZ71DOiC6Vp/jN1rsHp7hWcxVnS3MVfVOqqDO0tzccjUv9CaFNY8cwEa52y7XQh0Lb0YFvuhe2uy/w0mgUCUSXStPwVx7PgBEcN55NHnDNuGpOcTX8A+H6fHZ6Op92qT9GwjmXZQVUU+UUioxFKhWMafCBhbtE1ijUhrMPBHWKwf0cZuFOo1H7fIbF5std1UihRiDJalQQ5KbvNpzii2BbdUtjlQHsBLah4lpZ84Gw/noOuOfvJqdYaQcro5wyC21V2iIFVk4NVVAx7/QgtW8PyPEQJK9dTGF6UccHBO9+mCzXO4eb+TyXEHV2uYehVnk8S1tuD6y8C2zTMqqFYh8Z67+jW+nOeR2oKLEZ7dDlLqJtXZ7XwdTA2h2CtcoT1nbqYB/qnnGm58v5KQbKYpNrKGufUealL6lTZSqLGGyxLiF5TVIqTruPhHIM3l/HWJZZhrI0k5bbtmdmENsdwsDcEog90x8J4perGdTz4vedrOxcHrBkRWbl3CMJktsPAvt2nB01xaxjHwnmbxDBnOzk5NLAkLnMyU6UBoZukNa3B51xxJ6kBqTyh2J8W4T6R9TlISwNf2qzWl4A/78/+emLHRYXCoNRTqvjMEtgjNMxTuDgwTjFw1Alq3Hg2I3TOTCJcykOlbA3t599dKjH1qSIEr8+n/zrjgFtm05Aimt5hlXsOopTt/7ozoHqnyerydlmcpUakNbF9vjE7HxzHhkcNxv/Do+KhdttSte75i7Fvg2mGa4Va595FutcpjLpEvZWX+4Ift+TEp4//3PkNe5PCb7kcvskhG82V0Fk90wGxqTGJ5kMfMRdmDIqIjWia+aL+9Q5tBeL7lGTQaq4ZvnAmVW3EsL+0lFUxs96hl0IHzkc5zViX/lg0jn0pcZ5BpzwGPsEf/iUDJ/Cea18MkMyMQ5nlTVMwRuHhwxUsJ2E7PcVDd+BCOiU2/4oAW1QAo+HHiLJJAAVhAYiPI4ZCP2r3ZjH940uUVcTjuHxW+ZMDATQEIDxO4+d3o0Avl6/x7XFrF/4JbImMra6yHOScUHp2EhDJFNAyog3rEKeWiFKky54/G/QA53SKzrPWO2I/+1pk79Nz5cn8PRQvtzUMQPoZ6F1ec/yAR4+LAQFfZH8wnAhtSuSKd2mw6H21E8p+YkgJsFj9xnuBnuxW9Be5FJp04lCDgjGP4wYCUdnY3PTUjcjo79QJtbY3J44W5HpZKdyg/roFv3lbN+RymvUcc9g0H+AeqZQM72SBE/nnwtJ8NxLMdLEEZyK1DewjHKOjoYfFEo3khC5KaDke0lCHUYSP0zXa9gf71B4cTE9hVQ5C9h+CY10Mt+cBpf02dtZjCD26JaMQ49wkU1R/F+Ghi05OAWnNL8Xs8FBiIjxEJDoWK5sc7xZ3ipCbos1/jKbXi4m5/0qEHQrRBsSoBvafrAwPWGcCz88jvPD+sCNWc3QVF9lrWIcXjI2jg1KQmvVWKwZin4Q4eQMVmv4hsWDox3KIFa6lUvzXP5zMJEk2dCEofsjFDTL8GjRIdAFJrS5DO2xNRBOSgY9pTCbRieqUGjQBkJgpYg39KBiFn3KxjL4t5yh/xp8mZSlVFaGBjzMjUPfTXSoRfdoKcZiMFE6EY7fM1+v0I1RHenmyeJyhmB3JI2nk9UxmpD1LGhYaF30ZazdV+PUjwVtkqP9gftExVnKX1v04gsiRIQ+5EWyZHnDE1V9eqonwnnHnAjPTYNUn6Hw769WZ1d/ICK22EEdU1zdQw47jNSS/5DNdGSXLVLfT+noi6ePHGGbhyPLXrNeVaeRQ/BVZahmmXHOm5HIA7xd4LXJJrB9gpuz1AdBK3kccx1Clog3pXyFHed5RbASzQBvJ3hTaNL1CW/pdXII3+Y+oLw5gWXtHr6dDMDuTDpktXtI0ulJmIQ5OQxKGOrZyop1+8G0isI42FPjdufSAcvd7YdyAFCbQxJIl29ny4vZgXAGyZozRQ4xmBSa0eV0FVwmZnuxix89pDxDavuANMTzzqerLw7DFo37ReJVFdqiJ2xdCdHCV+YDtp2OzuV58573le97+X52+cd0FcaNHwaxE+PcGNjqeKgZEDvMOUgQh9N0/YQ8HjvEJo3FsqwvhL8+2ZRDcAfoWllmzIe+0Pm8nI1F2zEwOhZ+0Lcd0rGCy32tq/rOx/42PZ+iC8h0id/j/61JV7L70wGK8Yv5rJojNzzX6XM/CPEuBzK0zBb0/riGOyzFljvdzg4xx8a5pzRGBiRpDh3uc7m+UgPSHZDOxRecu4dKrxfhfgjSooh2nKzKCVFWTsZa/umJ9Uv05Dy9W4tqjFdNYp3v7E0dWh4gM/C+Aih2+1lfJqBD7ONtpIiA+K19rH9ebs21nzoEon0JRJdIpS/1Msb0Au+rRw5vGUAlBN8Db6h07QvebZvyTvC6EoiWogSiRTnp3E8Z5SOGN+UZEro7u8MG8ew7Y3uHUGY1B70TzKLdOMqocnSin2rpx45yHvptrqPcKOH+UQ6ZZijodWeYWXNapkSnyzmpQVJ3OReTLC7JeoD5huj1ujPjpvPMx6UBxUlqN1KsKzFwbpdjLinjIPmDglr1YuyAqx6nw026hDNNUbu6Lz/4McMaIl1NoFoerni7xsJIpuJWkG3p9+aiEZ0yTDmgiT8G46pLiyiTuFcJ1g/MB4TEXk4X06byo4syLiU8vpw3kKWER+lBH3dB38mMvvn46LdTHR0owMhidatij5WD68YPyrtLxDsrbyv6Ud4HEMDT1eSP2R24n2fsC/S6FHgxNzB/t07mOf7Zg+i/Y7vzLtodFd02HwZAfxItc7EX/sYZZNUU5vZVkPvY4fYpjOLs4Yx+G9jfTHHbi6uuQPNQci1zj+Zt3b7FGMZ4GiQCjfmyY/tJifS7DGsl0e5SuWllX+NakUOM41qd7TDHMY0R/34ZCGPPSFaXTgg2A0GdiGNGm3mcAUu0D2zm+rpQpD9uhs+nDjZO6+1UVcfzoFKyotp+Ja5oeb1iOJRYrcjHhovtXNR0j4asGKoRTbUiXjfTv9OK20cw6WU12DTdo2ymLqcVnefkHrezyNNTc0Pv0fZzqOnXx8sFKWpssnaTtjcukJkL3J2qYmOPrtETlGnlJh47OSKceEQDFq22RA8o0YgFM4K9r0lCOFAdDsq5ii5wTWOqLWuoz8blMNCbKVNxSWzqglof0czpjbvgNBTO2jFV72Jwzh/kW/McrkEkW++qXXCNORzPqyYKx2dB4NtKU5N2qCNjttpChknnTJMtZEiIoX2OoluEOcii4iQZfxqM6rWyfg78MgbnCXm9CQ5zaYw81vTXwihkv707Gz+nMRa4pzKox85dOCiXC2Vcb9zFMnfpO3FXqqPZx1Vx5jOaHWGAtWrGeifJjsnRoD0c8qLS2ARKU7yhydgWC912tWHGciL7MYQbp8qE8xWDxQ09JtIbJZShG2LktMMBRkM2VIzLwMyWbOiEdpg6bjTZUDBrcTrR1LyWNmSwlSwz9AmFdBgtLqyjGgyt/XBzhj6hsUxKzM2WZEPtjTQCHF+xXtwQ22HWN9OKbIirmIbunSUbhmna4EpDnxBXwb5qy+gqXUUSFw2TZC8m/ePnwsSEujceFFs77y5ptOlitp8HIxEIkIgztsVW3DgHGrOUykFjjBlL7SqlnVDQR8S8EmAHp7BuxYGJ5iRO/YIpKMmhIaeW1lALjuE0sLZaCbIfruKGcQad7Meg0aAkiJmYmMp4cASnJC6FDSeKJeUpJ70Es3FP9jPKckgZR01c8CO8Xa1dxX/p59To7863JoBMg+9hMeOmKQvjID1K2rSmjwd/GguY1nY4iS/wA7lBB3bhvhQ6E/Zjc9/bTT4Pu5/9WFB2ihnflsFce+9bIpjjMnQEJRhcBU9RegkXY9eFNvtBu1jlBWUHWORgYeUoN4QGDk7alheF4QXWSMrsoFlnhGf8GvNJxhkSGJTbJVSaB1OR7RQYgUHjUm4A6xr0AZBkOzAek+imYtu8J5Vj2mOyFdkO0TSjpaKizKPtAPSqpbLMcwc+lVS0WNgd8FX6yVs8YtbLdQS5s9710XG9eHf709DPJyeT5U1OnAONjn1tt8EHs7AgaxoIbt6YsdqD85KFDny1awWihs8k8pyj7Q46tEWp2cjBUkTuy9Ed4JixmsY8SDRYf2QHiWYIzgvqvOGOPRRHtYNHi2OER2oH0YFBoYAF2QHHTKDaqh3QWSEUUrR8N8d8qGIljxAen/xIMJfRo1LVQsCH/i6V1gobKAyAsu1xEoPBuJNvhNK5o5hkffENnIPENzeeWPt1uZqf7uMbizyDCpEwJbf+AMvNKOXWxeepHaVXnPg0aF2JOIJTlTxGRyEF52fLOHkLsEDFJyx2s0RIQ7jWFgKmWO0Yhlbpwm+1lIw354P56ba8EzcJM23sVqey9GWondobxKdsUBe+3iNktJUke6jAsN5X7BP2QGtODZXHdf0g6MbIhKsdMlyCk6tZHW/EkU3YsLw2HGUQFFjPDCzUhYXy+Wom/UPYffs56Olyjoj75CY7DyLawsKpGSEE+yzaHdU+COJwBqpFy0rYopOSDURfCVsc4Ib545mi1l3o7gnnwZBNmArxNEM2EVCFlluyCTQS7EWyCRw4TkMa8Um0xC3WFh0eDoFQREOrTTTi8SBmVeslHeSI4XQT9AlTTmpqyiHqqZTeBoDiJggBamcdeRIIF3CmIk8C41Bt7brQcwwZaY8Ba3IIz3cpGszReab6YyOV2cjf2E97Cn8KDtSTNyEcvyAH73byFRCDaJVcC+ISwTQJfFCHlXECC1JhywU2XIK777YGi4tfFOhPpmXFVnEPyGB0NauZEtcgyKkfpIIpidiZJbuEz2/9FxfvBaRolKn4Ku4iOWJ/lu4ivNFKkU0sXCLG6aOgq57dknzaBG9JVue44iaIL+LMDSebOG6R+uL0UbxH8ldXu+DXCQEIugtcIzCvHqISHco2Za7n8+zBHKPbCzmfblYbDC+4yU9Ce3mmeZ2exRXleR2GUyGrxQj1wzYzqrb/FZqJ8FaKKzSxRwC+VhQhF6xrByk0tmYknhFGG3Bbr42aE2Y19Y100Ki8dvPxKVfF73W8Yhzz9L4d53VSK/hwQlO3CHfAGMn/hab7Qij6myCaXutpLZAj9LXThZbcnKshktDltC9L/IJio96LJcz9R7Fh1ckFKiiuz+lqrPvgaZNiBxJuA73TsgLpSVWB5+3aiS0viPSBarV6r7iaELQyg5ZN4H1ah0FiZVibLuY5WUz5dskE97ROxJHVGK0a4XS1nlrRP3Y+kCmg5h4gmu3vzQbPJmgW98t0dbqXD4JHOyZZJVnxQSg6GJva1gmeeF1tUI7/VKsZTlaTnqxmHVmNxIyhLVpuDIJVmqxGFJ6sSDeuRgKCabXaX1EVF8bVHFnMG7KYJGUVsEM/Kb+k8wg/pA1EVTuHhrXCHIWLOcB8wDTOsEqXerk5xuf+vtOz57n+lcdx6LGdBytVzv6GHHnaeTeg0t9tPF9e6xqi4fqfAVFrgGqFqFAPj+hmhd/yfNfoa1YmqTZzdoCkLx1abrDZBkir6WrmyOotpEiGHwrpDUPYwqGFxa7QtQsx5lSeLssBFCuiZRUqllGedkOl5gDnFk7NKZzePBycP07OZ8c7Za0c5zy2jHPL0yjrMgCX+wHKLlAahnOjWygl0w8I5WyzS8La0h7YFaZEzECXvqN8UJpdcHSSsKQU7uFwfPEe8nUXR5ownS5hidIPBOvT2DHPS2dKg9CHvqG9xoDoNm3FNUVU+4dD9OXVarc5K1j07oP5Y9LA8dBxgfPYwyyYQEJfc9gHI2g3oEJRQM0DsuirzWK23gmps+UgJgye3ELDJXclGUF6cFM6wamoTYu6goeDc1/NPHjSpXhIGIcii0WbD9Yaf83nHHhzJ5iSYsmdfDgsn6Ed2fJ0t7hFsRRL+tPb0ggF9V7Z3/TISZjBrO2Ap6LWEL+WW+sTTxyU3mXWhkNQtoy9cCkaFLqu+9J/3Qx6swuW2lJnkz0gb942rbfdht3GHpChqatHcjR1dVWlqYmS1+YuP3Zdepe5ZK2on8pqdM/kZdQBsqOO0byfJxcYTPbj9P3omw36F/Q8lwxxQBdTaKWZTa4c9zneq33MieWgr/XpCEk/dHDfGWR/JjrIzSPZXjpQnelgutr0PT3ZxzPdqrSxyWcQbcZexOynMzk8LHyaq9uPw/sJgK9F7m60D3zmu4L/YomOFrNJ79OzI/6y9CDUMYPNmlxPlAy6GYAk0+mRgfm7JvNyLZTey/ymK/4vl3M0IEQ6YB2yAeu+dQBL7Sia1FDqBNH0tErndpvxhPFN1Y/p/imQgb9xui3IwHYng9DbJNWO9k0DKnK7K7JARJhZGeNg03kyagfoQRR0pAGX7ACtDqeBV0juzk4npxHmVziNmYz5Xk3CqBasLq1p40vmMilEq4CpZixeshkGUuhICj4NaTL8cFL4zz+Op7/P9oyjPYgCeOL+HCyPL1m0BE6KuuDF92fjdCzVDBTQjQJg9yUKkIdTwC+z6SVKPfq2C1WEVKsyCzOd5G/qRVLfqOwWSJ8aJPQT/nn88OuEvj/cKPx6gv54r2bnO8d3HeYXiGL5pyHGNrn9mf1Nap9W5kjo9FINfmHH7HYpdtW38v9NB8FR5HU52/TsCuCMTPQCLdOlu7UJ51ZhG+IE2rbMyOFdXHJNX3ODL4YP4UCrGzRB16wbywMbD6GEVr/D/mhBi3CyPRwn4ePkFeh4SjpdyclWbV2MIqFdTdQQocW90ok68H3mBrXQjRacyxXwB5DCN6vpdLFnWOdB+kCncxAi6wObOpy4MlEotSNTZa6Qiy/tJxgg6DQFruUPsAS8VIenCp5P5rNDJziqEgSSkvY0v14ffF9I7zvv7U8DqRQ3Bv7CuaKukKJ75/vJ1aFT7b2hg4N4mQ8Vor392G2PHlSdorla9BDGm2xwOvXJanJ86Fh0Pi5t6HkTtS8jCG6aHzQgWwdokwTWvQRoceOzw+dls2J9pxM3KR0nM7Da9WNgP3pgXZLDpgeWfTVZ/LEnA3c3WHWTa2vDavkAa6f4WULVHQ7q0+XlGtXfB8tgVQxjV2Lk1pbomBuEcKeoWDKCD8f0CX7pxeVscTinlpGazVxc0wS8Bky7YGpZnot7eKhzb+nTnfiUlYCmLsNwTXOcdcC0E6Ze5mmLh3Pq98v18t3yUDY1RaEKW/zVkpr0pp/g06PHNEcgeuDTH6bH4dc5WPY2kaasT1VThDQ4Nd2SDGXekjkktDxdHeyhOhUPcIRDACJGHcIhAFm8VPyl7cCl3SrH01ilQwLEt1WE3x9mlIfbBDN6n7qkXpELSmyLynLzKcYOO41Jo7I4mUzXtGuFbschabcbym/O8OmqeqTLmHndbh5hi2CWuh82fvXIAca4w3xg8gNgnLLE27hFB4i9LwHFgrAuqrcnV/bxI2z2HP6oLKneEA6ie34HfJluDnpkgKtSvn5k9KMH2CQWvj6a/AFYuLbBOsCrS9RCNvAW/mU9Hdt5/PDuywc8BLythEEHiGWJYqgCsSkpH8YHiLtAnI/ncfUBEH62Ob6DdJY0khzVcD52MdhXnYKOftdQsoeB9tsN+qSurrpzbhNStm3d25d/9Njh3RuregDbigSzunhHpcqCFcHsCvdyO9jOHXvrpSiHO5x9OzXV6wSsK+cgbHMsuuT3nBzEcrdS11zf+KDAplrYl5vTzjENHusWU6y5JIVEsZjlIJM7NvdKZcz8QbEtXb06oKpiu5kQjIx1NKGRRSyQCrHIsR1Q7VSQnDJ9Vj0kqN9Ml6uzzrAig1sSQ/xaeIrLT0vFdm+13g5P2V3hqXu3Wg/kEFqtd61RxkDR6WKKVuy7x3OE5WJTijwnDa9dOpwuyuvUhyKPVMe2LBan52bmeB2L1aVtxq+VJZnXZEmfZwyWJV0e+luWVHl0WVoyvN/M/kxLBsuA07tsL6lFtSTMfO/IktY3w2uaB69XhMkhXLUiCnd96yatIc/NdE9tf3995Hwg8pHs3rjA3IELnm7m8Bz3jVMLi0UClYWmE2k4oQltlHGFhRYwj5rQgtA1D5h0UNv9OyFgaykBW04I1uU5hA0P5LBxs2Q4FiLoXdoWBev2XVpKsarmgdSRhtEljWw9uPaUCZr305JW9FQm8NiZIM9B+ziq4NlqiQec3KwHrKQy2xNpaBt5mwmBkoH213SAI4RElEoQrpRUKaEa6an8hx8oyXJGUx3lCS+1FBSzVPSzcUuZeNnSJYqyJmXM9gns04Hkbzp/zcVHEfz0zPY+0V8LyUxohrdEvyXEYI2hhoV210S/FET0Wy8IudmG3rKt0lrSbOc0l7uUStG7bMlp7gU10jhlUIx0b4t+aVqiX5AVmeJkRcNsSz/ZQfJ3YgO5KwXyoUT/13Mcgpq/m5zewAfJGm6EXqZ7qanEpjTrpKQWkDNtDSCsbHESVSmSm5bJTgjMm2s+gGr5ANwRuW23Nn9a0FK3wm4HLZV7lLalCRx1ApRoOSr0qS0bfIBu2QS9K5vwoZjg+6vV2dUfpQBrPw+wxtrNUtW2eKClCxxrWcPbGcqNoG4xQXtJDJylNjvnVMzya2xgJXXYbcsVttq3+IpKbpunKtQPLqnS4y2DyLR8FS7Ic+v2iaqBDW5OqsmPYhJVabb9rvCWWhK1GW8ItRnXCrS4lj0kObWHQmtbQywNwSyhNmX5DZGWUKTu2/bQ9iuZgAXVXlxR3VLxRHbY2+aQNtQO5JyyhGaUB7xvKUDJ+0lQPWIeoJnHB/GF7zPgq0HdhlClEK62hDHaQikjVSUQrQn0VNtD1gZzuiYIdGsRTmtO9QFGK2Aot+O18W5D0toSU8aGE1SMkx1CTpTYMxjYIJ10lqoH7ItWx9JJuoNkjO6AuQ9cESbBUzEma8FuQ9MhqVveM76qrFSEPVzIE4it5Ag7YBK4ZUQV4Y6dtoI+g5K+r6Obj55zTAoiOfthOQetMUIz1PNp6Ya6j39EmD7hPKvJG9cw2J672uzBNcRYLK+tKXS6AvVJW1tUIkwBtyIf5W62wWAEzKP3xAySyEQ6rmwdZpLBtZbKGLqNhz9O7PjwOUR4laKsJNCcXznvamUlw3QGjWest8H0KovbrINGoZOTlFzVxI7vgoNtbt+z3QYTrwSmiNdCR2IGAO6RMJQMc2MlVWwyyB04JbW+xO1YMJ6zgzrqdn6L7cqpPzhTvZycL29kJhf6wTJVeyQu+AdC8FrMgnoQ3Kk5KTQIY4IY/WFSE0wk1fLRIaON57ImcKuxnmWaqjvoROIJB30HLeOpwkPOTbAWE7kwXFG4WlmET+qageLdSVanGV301kmyIcydCg8vKPu4MBCZaaJRoZ8FJwFsF/KYRtQmHG5DIwzN6c9kkf60ctBH3UqNkiXn5QdmndSc+3Z9JMMIeiY8laxIFXvlqTqSldUX1YRFnltwYqxAH8Ckk61MB64afNlYqvO85luPIW7iwTLca6qLBKeMGub7WCfamghFNcw5R/SqCllJzmrDFNe0cCQKhkvgGs/rTXDNGedbjBT1CxKhjv5ayN0bkhnHNYVRrvWvFdQVygIUUaphhrpkQ2Cg25nTVCdvP7ASKiXze0NimB0mYJ2QsJhzcH8sq0NEXMNWctQnQv9tFFszavDDhIWvrkUrTCZAU0Z44sPLYBAa39oFlpYhMVkoLI3PUv8ec5Vgo7lWElEHuvc0k8hlMOisIdvA7GTUOeKBpa11NKMeDlWrdkyZQUyg8yzJpuAal5rGv3ENKpnVzAomD3foPKk0YWEkqvD/F5KO/x8qz6piQYEBAA=='

train_raw = gzip.decompress(base64.b64decode(TRAIN_B64)).decode('utf-8')
train_cases = [json.loads(line) for line in train_raw.strip().splitlines() if line.strip()]
print(f'Unpacked {len(train_cases)} clean train samples (Expected: 2461).')

eval_raw = gzip.decompress(base64.b64decode(EVAL_B64)).decode('utf-8')
eval_cases = [json.loads(line) for line in eval_raw.strip().splitlines() if line.strip()]
print(f'Unpacked {len(eval_cases)} clean eval samples (Expected: 180).')

train_year_dist = Counter(int(x['year']) for x in train_cases)
eval_year_dist = Counter(int(x['year']) for x in eval_cases)
print('Train year distribution:', dict(sorted(train_year_dist.items())))
print('Eval year distribution:', dict(sorted(eval_year_dist.items())))
assert len(train_cases) == 2461 and len(eval_cases) == 180
assert eval_year_dist == {1947: 48, 1984: 72, 2022: 60}


In [ ]:
# --- 1. Ticktack Temporal Encoding Math (Han et al., 2025 Eq. 1-4) ---
def compute_canchi_cycle(year: int) -> int:
    """Gregorian year to 60-year sexagenary cycle index (Eq. 1)."""
    if year >= 4:
        return (year - 3) % 60
    elif year > 0:
        return (60 - abs(year - 3)) % 60
    else:
        return (60 - abs(year) - 2) % 60

def compute_batch_te(years: torch.Tensor, hidden_dim=896, scale_mode='raw', alpha=0.5, beta=0.5, device='cpu', dtype=torch.float32):
    """Computes vectorized TE in R^{B x 1 x hidden_dim} (Eq. 2, 3, 4)."""
    cycle_indices = ((years.to(torch.int64) - 3) % 60).to(torch.float32)
    theta = (cycle_indices * 6.0) * (math.pi / 180.0)
    r = alpha + beta * theta
    x = (r * torch.cos(theta)).view(-1, 1).to(device)
    y = (r * torch.sin(theta)).view(-1, 1).to(device)
    
    dim_indices = torch.arange(0, hidden_dim, 2, device=device, dtype=torch.float32)
    inv_freq = 1.0 / (10000.0 ** (dim_indices / float(hidden_dim))).unsqueeze(0)
    
    x_inp = x * inv_freq
    te_x = torch.stack([torch.sin(x_inp), torch.cos(x_inp)], dim=-1).flatten(start_dim=-2)
    
    y_inp = y * inv_freq
    te_y = torch.stack([torch.sin(y_inp), torch.cos(y_inp)], dim=-1).flatten(start_dim=-2)
    
    te_raw = (te_x + te_y).unsqueeze(1).to(dtype=dtype)  # [B, 1, D]
    if scale_mode == 'raw':
        return te_raw
    elif scale_mode == 'scaled':
        scale = 1.0 / float(hidden_dim)
        return (te_raw * scale).to(dtype=dtype)
    else:
        raise ValueError(f'Unknown scale_mode: {scale_mode}')

# --- 2. Standard LoRA Linear Layer (Hu et al., 2021) ---
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=8, alpha=16.0, bias=False):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / float(rank) if rank > 0 else 1.0
        self.base_layer = nn.Linear(in_features, out_features, bias=bias)
        self.base_layer.weight.requires_grad = False
        if self.base_layer.bias is not None:
            self.base_layer.bias.requires_grad = False
        self.lora_A = nn.Parameter(torch.empty(rank, in_features))
        self.lora_B = nn.Parameter(torch.empty(out_features, rank))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    @classmethod
    def from_linear(cls, linear, rank=8, alpha=16.0):
        mod = cls(linear.in_features, linear.out_features, rank=rank, alpha=alpha, bias=(linear.bias is not None))
        mod.base_layer.weight.data.copy_(linear.weight.data)
        mod.base_layer.weight.requires_grad = False
        if linear.bias is not None:
            mod.base_layer.bias.data.copy_(linear.bias.data)
            mod.base_layer.bias.requires_grad = False
        mod.to(device=linear.weight.device, dtype=linear.weight.dtype)
        return mod

    def forward(self, x):
        base_out = self.base_layer(x)
        if self.rank == 0:
            return base_out
        lora_out = F.linear(F.linear(x, self.lora_A.to(dtype=x.dtype, device=x.device)), self.lora_B.to(dtype=x.dtype, device=x.device)) * self.scaling
        return base_out + lora_out

def apply_lora_to_model(model, target_modules=['q_proj', 'v_proj'], rank=8, alpha=16.0):
    adapted = {}
    for name, module in model.named_modules():
        for t in target_modules:
            if name.endswith(t) and isinstance(module, nn.Linear):
                parts = name.split('.')
                parent = model
                for p in parts[:-1]:
                    parent = getattr(parent, p)
                layer_name = parts[-1]
                adapter = LoRALinear.from_linear(module, rank=rank, alpha=alpha)
                setattr(parent, layer_name, adapter)
                adapted[name] = adapter
    return adapted

# --- 3. Eq. 8 Intra-class Contrastive Loss with Can Chi Sexagenary Cycle Grouping ---
def compute_intra_loss(hidden_states: torch.Tensor, years: torch.Tensor):
    """Eq. 8 intra-class contrastive loss clustering by Can Chi sexagenary cycle (Han et al., 2025)."""
    B = hidden_states.size(0)
    norm_h = F.normalize(hidden_states, p=2, dim=-1)
    sim_matrix = torch.matmul(norm_h, norm_h.T)
    canchi_cycles = ((years.to(torch.int64) - 3) % 60).to(device=hidden_states.device)
    same_cycle_mask = (canchi_cycles.unsqueeze(1) == canchi_cycles.unsqueeze(0)) & (~torch.eye(B, dtype=torch.bool, device=hidden_states.device))
    if same_cycle_mask.sum() > 0:
        return 1.0 - sim_matrix[same_cycle_mask].mean()
    else:
        return (hidden_states * 0.0).sum()

print('Ticktack architecture math and LoRA modules defined successfully.')


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'

print(f'Loading tokenizer for {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

def load_base_model():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=REVISION,
        torch_dtype=torch.float32,
        device_map={'': device},
        trust_remote_code=True
    )
    for p in model.parameters():
        p.requires_grad = False
    model.eval()
    return model

def load_lora_model(rank=8, alpha=16.0):
    model = load_base_model()
    adapted = apply_lora_to_model(model, target_modules=['q_proj', 'v_proj'], rank=rank, alpha=alpha)
    return model, adapted

# Dataset for Next-Word Prediction
class JointNWPDataset(Dataset):
    def __init__(self, samples, tokenizer, max_len=128):
        self.examples = []
        for s in samples:
            inp = s['input']
            tgt = str(s['target'])
            if not tgt.startswith(' ') and not inp.endswith(' '):
                tgt = ' ' + tgt
            enc_p = tokenizer.encode(inp, add_special_tokens=False)
            enc_t = tokenizer.encode(tgt, add_special_tokens=False)
            ids = enc_p + enc_t
            lbls = [-100] * len(enc_p) + enc_t
            if len(ids) > max_len:
                ids = ids[:max_len]
                lbls = lbls[:max_len]
            self.examples.append({
                'input_ids': ids,
                'labels': lbls,
                'year': int(s['year'])
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

def collate_joint_fn(batch, pad_token_id):
    max_len = max(len(x['input_ids']) for x in batch)
    input_ids, labels, attention_mask, years = [], [], [], []
    for x in batch:
        pad_len = max_len - len(x['input_ids'])
        input_ids.append(x['input_ids'] + [pad_token_id] * pad_len)
        labels.append(x['labels'] + [-100] * pad_len)
        attention_mask.append([1] * len(x['input_ids']) + [0] * pad_len)
        years.append(x['year'])
    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        'year': torch.tensor(years, dtype=torch.long)
    }

# --- Date Token Masking & Localized TE Injection ---
def find_date_token_mask(input_ids: torch.Tensor, years: torch.Tensor, tokenizer, device=None):
    """Finds boolean mask [B, S, 1] identifying exact date token sub-sequences."""
    if device is None:
        device = input_ids.device
    B, S = input_ids.shape
    mask = torch.zeros(B, S, 1, dtype=torch.bool, device=device)
    for b in range(B):
        yr = int(years[b].item())
        sub = tokenizer.encode(str(yr), add_special_tokens=False)
        seq = input_ids[b].tolist()
        sub_len = len(sub)
        found = False
        for i in range(len(seq) - sub_len + 1):
            if seq[i:i+sub_len] == sub:
                mask[b, i:i+sub_len, 0] = True
                found = True
        if not found:
            mask[b, 1:min(5, S), 0] = True
    return mask

def inject_te_embeddings(model, tokenizer, input_ids: torch.Tensor, years: torch.Tensor, te_mode='broadcast', scale_mode='raw', device=None):
    """Injects TE according to te_mode ('none', 'broadcast', 'date_token')."""
    if device is None:
        device = input_ids.device
    token_embeds = model.model.embed_tokens(input_ids)
    if te_mode == 'none':
        return token_embeds
    te_batch = compute_batch_te(years, hidden_dim=model.config.hidden_size, scale_mode=scale_mode, device=device, dtype=model.dtype)
    if te_mode == 'broadcast':
        return token_embeds + te_batch
    elif te_mode == 'date_token':
        mask = find_date_token_mask(input_ids, years, tokenizer, device=device)
        return torch.where(mask, token_embeds + te_batch, token_embeds)
    else:
        raise ValueError(f'Unknown te_mode: {te_mode}')

# --- Option Permutation Generator (Letter Bias Debiasing) ---
def generate_option_permutations(prompt_text: str, target_text: str):
    """
    Generates option permutations to evaluate and debias letter / ordering bias.
    - 4-choice MCQA: 4 cyclic rotations (Rot 0: ABCD, Rot 1: BCDA, Rot 2: CDAB, Rot 3: DABC)
      and transforms target accordingly.
    - Yes/No: 2 orderings ('Answer as Yes or No', 'Answer as No or Yes').
    """
    tgt = target_text.strip().lower()
    if tgt in ['yes', 'no']:
        p0 = prompt_text
        p1 = re.sub(r'Answer as Yes or No', 'Answer as No or Yes', prompt_text)
        return [
            {'prompt': p0, 'target': tgt, 'rot_idx': 0, 'rot_name': 'Yes/No'},
            {'prompt': p1, 'target': tgt, 'rot_idx': 1, 'rot_name': 'No/Yes'}
        ]
    suffix = ''
    m_suffix = re.search(r'(\s*\n?\s*Answer:\s*)$', prompt_text, re.IGNORECASE)
    if m_suffix:
        suffix = m_suffix.group(1)
        base_prompt = prompt_text[:len(prompt_text) - len(suffix)]
    else:
        base_prompt = prompt_text
    pattern = re.compile(r'^(.*Options:\s*)\(a\)\s*(.*?)\s*\(b\)\s*(.*?)\s*\(c\)\s*(.*?)\s*\(d\)\s*(.*?)$', re.DOTALL)
    m = pattern.search(base_prompt)
    if not m:
        return [{'prompt': prompt_text, 'target': tgt, 'rot_idx': 0, 'rot_name': 'orig'}]
    prefix = m.group(1)
    opts = [m.group(2).strip(), m.group(3).strip(), m.group(4).strip(), m.group(5).strip()]
    letters = ['a', 'b', 'c', 'd']
    if tgt not in letters:
        return [{'prompt': prompt_text, 'target': tgt, 'rot_idx': 0, 'rot_name': 'orig'}]
    t_idx = letters.index(tgt)
    rot_names = ['ABCD', 'BCDA', 'CDAB', 'DABC']
    permutations = []
    for k in range(4):
        rot_opts = [opts[(j + k) % 4] for j in range(4)]
        rot_target = letters[(t_idx - k) % 4]
        rot_prompt = f'{prefix}(a) {rot_opts[0]} (b) {rot_opts[1]} (c) {rot_opts[2]} (d) {rot_opts[3]}{suffix}'
        permutations.append({
            'prompt': rot_prompt,
            'target': rot_target,
            'rot_idx': k,
            'rot_name': rot_names[k]
        })
    return permutations

# --- Academic Choice Logit Evaluator with Rotation Support ---
def eval_choice_logit_single(model, tokenizer, prompt_text: str, target_text: str, year: int = None, te_mode: str = 'none', te_scale_mode: str = 'raw'):
    prompt = f'Question: {prompt_text.strip()}\nAnswer:' if not prompt_text.strip().endswith('Answer:') else prompt_text.strip()
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        if te_mode != 'none' and year is not None:
            y_t = torch.tensor([year], dtype=torch.long, device=device)
            inputs_embeds = inject_te_embeddings(model, tokenizer, inputs['input_ids'], y_t, te_mode=te_mode, scale_mode=te_scale_mode, device=device)
            out = model(inputs_embeds=inputs_embeds, attention_mask=inputs['attention_mask'])
        else:
            out = model(**inputs)
        logits = out.logits[0, -1, :]
    tgt = target_text.strip().lower()
    if tgt in ['yes', 'no']:
        scores = {
            'yes': max(logits[7414].item(), logits[9454].item(), logits[9834].item(), logits[9693].item()),
            'no': max(logits[2308].item(), logits[2753].item(), logits[902].item(), logits[2152].item()),
        }
        pred = 'yes' if scores['yes'] >= scores['no'] else 'no'
    else:
        scores = {
            'a': max(logits[264].item(), logits[362].item(), logits[64].item(), logits[32].item()),
            'b': max(logits[293].item(), logits[425].item(), logits[65].item(), logits[33].item()),
            'c': max(logits[272].item(), logits[356].item(), logits[66].item(), logits[34].item()),
            'd': max(logits[294].item(), logits[422].item(), logits[67].item(), logits[35].item()),
        }
        pred = max(scores, key=scores.get)
    top1_id = logits.argmax().item()
    top1_str = tokenizer.decode([top1_id]).strip()
    is_correct = int(pred == tgt)
    greedy_is_correct = int(top1_str.lower() == tgt)
    return {
        'prediction': pred,
        'is_valid': 1,
        'is_correct': is_correct,
        'greedy_top1': top1_str,
        'greedy_is_correct': greedy_is_correct,
        'scores': {k: round(v, 4) for k, v in scores.items()}
    }

def evaluate_dataset(model, tokenizer, eval_cases, te_mode='none', te_scale_mode='raw', debias=True):
    model.eval()
    total = len(eval_cases)
    case_results = {}
    canonical_correct = 0
    debiased_mean_correct_sum = 0.0
    debiased_strict_correct = 0
    year_stats = defaultdict(lambda: {'total': 0, 'canonical_correct': 0, 'debiased_mean_sum': 0.0, 'debiased_strict': 0})
    canonical_pred_counts = Counter()
    rot_pred_counts = defaultdict(Counter)
    rot_correct_counts = defaultdict(lambda: {'total': 0, 'correct': 0})
    
    for c in eval_cases:
        yr = int(c['year'])
        perms = generate_option_permutations(c['input'], c['target']) if debias else [{'prompt': c['input'], 'target': c['target'], 'rot_idx': 0, 'rot_name': 'orig'}]
        perm_results = []
        for p in perms:
            res = eval_choice_logit_single(model, tokenizer, p['prompt'], p['target'], year=yr, te_mode=te_mode, te_scale_mode=te_scale_mode)
            perm_results.append({
                'rot_idx': p['rot_idx'],
                'rot_name': p['rot_name'],
                'target': p['target'],
                'prediction': res['prediction'],
                'is_correct': res['is_correct'],
                'greedy_top1': res['greedy_top1'],
                'scores': res['scores']
            })
            rot_pred_counts[p['rot_name']][res['prediction']] += 1
            rot_correct_counts[p['rot_name']]['total'] += 1
            rot_correct_counts[p['rot_name']]['correct'] += res['is_correct']
        
        canonical_res = perm_results[0]
        canonical_is_corr = canonical_res['is_correct']
        canonical_pred = canonical_res['prediction']
        canonical_correct += canonical_is_corr
        canonical_pred_counts[canonical_pred] += 1
        
        sample_mean_corr = sum(r['is_correct'] for r in perm_results) / float(len(perm_results))
        sample_strict_corr = int(all(r['is_correct'] for r in perm_results))
        debiased_mean_correct_sum += sample_mean_corr
        debiased_strict_correct += sample_strict_corr
        
        year_stats[yr]['total'] += 1
        year_stats[yr]['canonical_correct'] += canonical_is_corr
        year_stats[yr]['debiased_mean_sum'] += sample_mean_corr
        year_stats[yr]['debiased_strict'] += sample_strict_corr
        
        case_results[c['example_id']] = {
            'year': yr,
            'target': c['target'],
            'prediction': canonical_pred,
            'is_correct': canonical_is_corr,
            'greedy_top1': canonical_res['greedy_top1'],
            'debiased_mean_acc': sample_mean_corr,
            'is_consistent': sample_strict_corr,
            'rotations': perm_results
        }
        
    canonical_acc = (canonical_correct / float(total)) * 100.0
    debiased_mean_acc = (debiased_mean_correct_sum / float(total)) * 100.0
    debiased_strict_acc = (debiased_strict_correct / float(total)) * 100.0
    
    by_year = {}
    for yr, st in sorted(year_stats.items()):
        cnt = st['total']
        by_year[yr] = {
            'canonical_acc': (st['canonical_correct'] / float(cnt)) * 100.0 if cnt > 0 else 0.0,
            'debiased_mean_acc': (st['debiased_mean_sum'] / float(cnt)) * 100.0 if cnt > 0 else 0.0,
            'debiased_strict_acc': (st['debiased_strict'] / float(cnt)) * 100.0 if cnt > 0 else 0.0,
        }
        
    rot_accuracies = {}
    for rname, rdata in rot_correct_counts.items():
        rot_accuracies[rname] = (rdata['correct'] / float(rdata['total'])) * 100.0 if rdata['total'] > 0 else 0.0
        
    return {
        'overall_acc': canonical_acc,
        'canonical_acc': canonical_acc,
        'debiased_mean_acc': debiased_mean_acc,
        'debiased_strict_acc': debiased_strict_acc,
        'invalid_rate': 0.0,
        'by_year': by_year,
        'case_results': case_results,
        'pred_counts': dict(canonical_pred_counts),
        'rot_pred_counts': {k: dict(v) for k, v in rot_pred_counts.items()},
        'rot_accuracies': rot_accuracies
    }

# --- Unified Configuration Training & Evaluation Runner ---
EPOCHS = 2
BATCH_SIZE = 8
GRAD_ACCUM = 2
LR = 1e-4
WEIGHT_DECAY = 0.01
SEEDS = [1, 2, 3]

def train_and_eval_configuration(cfg_name, te_mode='broadcast', te_scale_mode='raw', sigma=0.5, seeds=SEEDS):
    results = []
    pad_id = tokenizer.pad_token_id
    print('=' * 60)
    print(f'RUNNING: {cfg_name.upper()} (te_mode={te_mode}, scale={te_scale_mode}, sigma={sigma})')
    print('=' * 60)
    for seed_idx, seed in enumerate(seeds):
        print(f'\n--- Training {cfg_name}: Seed {seed} ({seed_idx+1}/{len(seeds)}) ---')
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        model, adapted = load_lora_model(rank=8, alpha=16.0)
        optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
        ds = JointNWPDataset(train_cases, tokenizer)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda b: collate_joint_fn(b, pad_id))
        model.train()
        t_start = time.time()
        for epoch in range(EPOCHS):
            epoch_loss, epoch_ntp, epoch_intra, step_cnt = 0.0, 0.0, 0.0, 0
            optimizer.zero_grad()
            for step, batch in enumerate(loader):
                b_in = batch['input_ids'].to(device)
                b_lbl = batch['labels'].to(device)
                b_msk = batch['attention_mask'].to(device)
                b_yr = batch['year'].to(device)
                if te_mode != 'none':
                    inputs_embeds = inject_te_embeddings(model, tokenizer, b_in, b_yr, te_mode=te_mode, scale_mode=te_scale_mode, device=device)
                    out = model(inputs_embeds=inputs_embeds, labels=b_lbl, attention_mask=b_msk, output_hidden_states=True)
                else:
                    out = model(input_ids=b_in, labels=b_lbl, attention_mask=b_msk, output_hidden_states=True)
                loss_ntp = out.loss
                if sigma > 0.0:
                    seq_lengths = b_msk.sum(dim=1) - 1
                    last_hidden = out.hidden_states[-1][torch.arange(b_in.size(0), device=device), seq_lengths, :]
                    loss_intra = compute_intra_loss(last_hidden, b_yr)
                    loss_total = loss_ntp + sigma * loss_intra
                else:
                    loss_intra = torch.tensor(0.0, device=device)
                    loss_total = loss_ntp
                (loss_total / GRAD_ACCUM).backward()
                epoch_loss += loss_total.item()
                epoch_ntp += loss_ntp.item()
                epoch_intra += loss_intra.item()
                step_cnt += 1
                if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(loader):
                    optimizer.step()
                    optimizer.zero_grad()
            avg_loss = epoch_loss / step_cnt
            avg_ntp = epoch_ntp / step_cnt
            avg_intra = epoch_intra / step_cnt
            print(f'  Epoch {epoch+1}/{EPOCHS} - Total: {avg_loss:.4f} (NTP: {avg_ntp:.4f}, Intra: {avg_intra:.4f})')
        res = evaluate_dataset(model, tokenizer, eval_cases, te_mode=te_mode, te_scale_mode=te_scale_mode, debias=True)
        dur = time.time() - t_start
        print(f'  Seed {seed} Result: Canonical ACC = {res["canonical_acc"]:.2f}%, Debiased Mean ACC = {res["debiased_mean_acc"]:.2f}%, Consistency = {res["debiased_strict_acc"]:.2f}% (Time: {dur:.1f}s)')
        results.append({
            'seed': seed,
            'overall_acc': res['overall_acc'],
            'canonical_acc': res['canonical_acc'],
            'debiased_mean_acc': res['debiased_mean_acc'],
            'debiased_strict_acc': res['debiased_strict_acc'],
            'by_year': res['by_year'],
            'case_results': res['case_results'],
            'pred_counts': res['pred_counts'],
            'rot_pred_counts': res['rot_pred_counts'],
            'rot_accuracies': res['rot_accuracies'],
            'duration': dur
        })
        del model, adapted, optimizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return results

print('Model loaders, Debiasing Engine, and Unified Runner configured successfully.')


In [ ]:
print('========================================================')
print('CONFIGURATION 0: BASE MODEL ZERO-SHOT (ANCHOR BASELINE)')
print('========================================================')

t0_base = time.time()
base_model = load_base_model()
base_res = evaluate_dataset(base_model, tokenizer, eval_cases, te_mode='none', debias=True)
base_time = time.time() - t0_base

print(f'Base Model Evaluation Completed in {base_time:.2f}s:')
print(f'  Canonical Accuracy:       {base_res["canonical_acc"]:.2f}% ({sum(r["is_correct"] for r in base_res["case_results"].values())}/{len(eval_cases)})')
print(f'  Debiased Mean Accuracy:   {base_res["debiased_mean_acc"]:.2f}%')
print(f'  Consistency (Strict Acc): {base_res["debiased_strict_acc"]:.2f}%')
print(f'  Canonical Prediction Dist: {base_res["pred_counts"]}')

del base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
print('>>> Executing Control 1: Raw Broadcast (Paper Formula, sigma = 0.5) <<<')
c1_results = train_and_eval_configuration('C1_raw_broadcast', te_mode='broadcast', te_scale_mode='raw', sigma=0.5, seeds=SEEDS)


In [ ]:
print('>>> Executing Control 2: Raw Date-token only (Localized Injection, sigma = 0.5) <<<')
c2_results = train_and_eval_configuration('C2_raw_date_token', te_mode='date_token', te_scale_mode='raw', sigma=0.5, seeds=SEEDS)


In [ ]:
print('>>> Executing Control 3: Raw with sigma = 0 (Pure NTP, no intra loss) <<<')
c3_results = train_and_eval_configuration('C3_raw_sigma0', te_mode='broadcast', te_scale_mode='raw', sigma=0.0, seeds=SEEDS)


In [ ]:
print('>>> Executing Control 4: PT + Intra loss (No TE, sigma = 0.5) <<<')
c4_results = train_and_eval_configuration('C4_pt_intra_loss', te_mode='none', te_scale_mode='none', sigma=0.5, seeds=SEEDS)


In [ ]:
print('========================================================')
print('FINAL STATISTICAL AUDIT & COMPARATIVE CONTROL SUMMARY')
print('========================================================')

def compute_mcnemar_table(cases_a, cases_b):
    b = sum(1 for cid in cases_a if cases_a[cid]['is_correct'] and not cases_b[cid]['is_correct'])
    c = sum(1 for cid in cases_a if not cases_a[cid]['is_correct'] and cases_b[cid]['is_correct'])
    a = sum(1 for cid in cases_a if cases_a[cid]['is_correct'] and cases_b[cid]['is_correct'])
    d = sum(1 for cid in cases_a if not cases_a[cid]['is_correct'] and not cases_b[cid]['is_correct'])
    diff = abs(b - c)
    chi2_edwards = ((diff - 1.0) ** 2) / (b + c) if (b + c) > 0 and diff >= 1 else 0.0
    p_edwards = stats.chi2.sf(chi2_edwards, df=1) if chi2_edwards > 0 else 1.0
    p_exact = stats.binomtest(min(b, c), b + c, 0.5).pvalue if (b + c) > 0 else 1.0
    return {
        'contingency': {'a_both_correct': a, 'b_only_a': b, 'c_only_b': c, 'd_both_incorrect': d},
        'chi2_edwards': float(chi2_edwards),
        'p_value_edwards': float(p_edwards),
        'p_value_exact': float(p_exact)
    }

def get_metrics_summary(results_list):
    c_accs = [r['canonical_acc'] for r in results_list]
    d_accs = [r['debiased_mean_acc'] for r in results_list]
    s_accs = [r['debiased_strict_acc'] for r in results_list]
    return {
        'canonical_mean': float(np.mean(c_accs)),
        'canonical_std': float(np.std(c_accs, ddof=1)),
        'debiased_mean': float(np.mean(d_accs)),
        'debiased_mean_std': float(np.std(d_accs, ddof=1)),
        'consistency_mean': float(np.mean(s_accs)),
        'consistency_std': float(np.std(s_accs, ddof=1)),
        'seeds_canonical': c_accs,
        'seeds_debiased': d_accs,
        'seeds_consistency': s_accs,
    }

summary = {
    'base': {
        'canonical_acc': float(base_res['canonical_acc']),
        'debiased_mean_acc': float(base_res['debiased_mean_acc']),
        'consistency': float(base_res['debiased_strict_acc']),
        'by_year': {str(k): float(v['canonical_acc']) for k, v in base_res['by_year'].items()},
    },
    'c1_raw_broadcast': get_metrics_summary(c1_results),
    'c2_raw_date_token': get_metrics_summary(c2_results),
    'c3_raw_sigma0': get_metrics_summary(c3_results),
    'c4_pt_intra_loss': get_metrics_summary(c4_results),
}

print('--- SUMMARY OF 4-WAY CONTROL MATRIX BENCHMARK ---')
print(f"Base (Zero-shot):       Canonical = {summary['base']['canonical_acc']:.2f}%, Debiased = {summary['base']['debiased_mean_acc']:.2f}%, Consistency = {summary['base']['consistency']:.2f}%")
print(f"C1 (Raw Broadcast):     Canonical = {summary['c1_raw_broadcast']['canonical_mean']:.2f}±{summary['c1_raw_broadcast']['canonical_std']:.2f}%, Debiased = {summary['c1_raw_broadcast']['debiased_mean']:.2f}±{summary['c1_raw_broadcast']['debiased_mean_std']:.2f}%, Consistency = {summary['c1_raw_broadcast']['consistency_mean']:.2f}±{summary['c1_raw_broadcast']['consistency_std']:.2f}%")
print(f"C2 (Date-token only):   Canonical = {summary['c2_raw_date_token']['canonical_mean']:.2f}±{summary['c2_raw_date_token']['canonical_std']:.2f}%, Debiased = {summary['c2_raw_date_token']['debiased_mean']:.2f}±{summary['c2_raw_date_token']['debiased_mean_std']:.2f}%, Consistency = {summary['c2_raw_date_token']['consistency_mean']:.2f}±{summary['c2_raw_date_token']['consistency_std']:.2f}%")
print(f"C3 (Raw sigma=0):       Canonical = {summary['c3_raw_sigma0']['canonical_mean']:.2f}±{summary['c3_raw_sigma0']['canonical_std']:.2f}%, Debiased = {summary['c3_raw_sigma0']['debiased_mean']:.2f}±{summary['c3_raw_sigma0']['debiased_mean_std']:.2f}%, Consistency = {summary['c3_raw_sigma0']['consistency_mean']:.2f}±{summary['c3_raw_sigma0']['consistency_std']:.2f}%")
print(f"C4 (PT + Intra loss):   Canonical = {summary['c4_pt_intra_loss']['canonical_mean']:.2f}±{summary['c4_pt_intra_loss']['canonical_std']:.2f}%, Debiased = {summary['c4_pt_intra_loss']['debiased_mean']:.2f}±{summary['c4_pt_intra_loss']['debiased_mean_std']:.2f}%, Consistency = {summary['c4_pt_intra_loss']['consistency_mean']:.2f}±{summary['c4_pt_intra_loss']['consistency_std']:.2f}%")

# Paired McNemar Tests across Seed 1
mcnemar_tests = {
    'c1_vs_base': compute_mcnemar_table(c1_results[0]['case_results'], base_res['case_results']),
    'c2_vs_base': compute_mcnemar_table(c2_results[0]['case_results'], base_res['case_results']),
    'c2_vs_c1':   compute_mcnemar_table(c2_results[0]['case_results'], c1_results[0]['case_results']),
    'c3_vs_c1':   compute_mcnemar_table(c3_results[0]['case_results'], c1_results[0]['case_results']),
    'c4_vs_base': compute_mcnemar_table(c4_results[0]['case_results'], base_res['case_results']),
    'c4_vs_c1':   compute_mcnemar_table(c4_results[0]['case_results'], c1_results[0]['case_results']),
}

print('\n--- PAIRED MCNEMAR AUDIT (SEED 1) ---')
for k, v in mcnemar_tests.items():
    print(f"{k}: Contingency={v['contingency']}, Chi2={v['chi2_edwards']:.3f}, p_edwards={v['p_value_edwards']:.4e}, p_exact={v['p_value_exact']:.4e}")

def compute_pooled_mcnemar_table(runs_a, runs_b):
    b, c, a, d = 0, 0, 0, 0
    for r_a, r_b in zip(runs_a, runs_b):
        cases_a = r_a['case_results']
        cases_b = r_b['case_results']
        for cid in cases_a:
            ca = bool(cases_a[cid]['is_correct'])
            cb = bool(cases_b[cid]['is_correct'])
            if ca and not cb: b += 1
            elif not ca and cb: c += 1
            elif ca and cb: a += 1
            else: d += 1
    diff = abs(b - c)
    chi2_edwards = ((diff - 1.0) ** 2) / (b + c) if (b + c) > 0 and diff >= 1 else 0.0
    p_edwards = stats.chi2.sf(chi2_edwards, df=1) if chi2_edwards > 0 else 1.0
    p_exact = stats.binomtest(min(b, c), b + c, 0.5).pvalue if (b + c) > 0 else 1.0
    return {
        'contingency': {'a_both_correct': a, 'b_only_a': b, 'c_only_b': c, 'd_both_incorrect': d},
        'chi2_edwards': float(chi2_edwards),
        'p_value_edwards': float(p_edwards),
        'p_value_exact': float(p_exact)
    }

base_runs_mock = [{'case_results': base_res['case_results']} for _ in range(len(SEEDS))]
pooled_mcnemar_tests = {
    'c1_vs_base_pooled': compute_pooled_mcnemar_table(c1_results, base_runs_mock),
    'c2_vs_base_pooled': compute_pooled_mcnemar_table(c2_results, base_runs_mock),
    'c2_vs_c1_pooled':   compute_pooled_mcnemar_table(c2_results, c1_results),
    'c3_vs_c1_pooled':   compute_pooled_mcnemar_table(c3_results, c1_results),
    'c4_vs_base_pooled': compute_pooled_mcnemar_table(c4_results, base_runs_mock),
    'c4_vs_c1_pooled':   compute_pooled_mcnemar_table(c4_results, c1_results),
}

print('\n--- POOLED PAIRED MCNEMAR AUDIT (ALL 3 SEEDS, N=540) ---')
for k, v in pooled_mcnemar_tests.items():
    print(f"{k}: Contingency={v['contingency']}, Chi2={v['chi2_edwards']:.3f}, p_edwards={v['p_value_edwards']:.4e}, p_exact={v['p_value_exact']:.4e}")

# Letter Bias Audit: Percentage of 'a' predictions on MCQA
letter_bias_audit = {}
for name, runs in [('C1_raw_broadcast', c1_results), ('C2_raw_date_token', c2_results), ('C3_raw_sigma0', c3_results), ('C4_pt_intra_loss', c4_results)]:
    pct_a = [r['pred_counts'].get('a', 0) / float(len(eval_cases)) * 100.0 for r in runs]
    letter_bias_audit[name] = {'pct_a_mean': float(np.mean(pct_a)), 'pct_a_seeds': pct_a}
letter_bias_audit['base'] = {'pct_a': base_res['pred_counts'].get('a', 0) / float(len(eval_cases)) * 100.0}
print('\n--- LETTER BIAS AUDIT (% PREDICTED OPTION "A") ---')
for k, v in letter_bias_audit.items():
    print(f"{k}: {v}")

final_export = {
    'metadata': {
        'model': MODEL_ID,
        'benchmark': 'Ticktack 4-Way Disentanglement Control Matrix + Debiased Permutations',
        'evaluation_protocol': 'Choice Logit Scoring with 4-fold Permutation Rotation (Letter Bias Debiasing)',
        'seeds': SEEDS,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LR,
        'sigma': 0.5
    },
    'summary': summary,
    'mcnemar_audit': mcnemar_tests,
    'pooled_mcnemar_audit': pooled_mcnemar_tests,
    'letter_bias_audit': letter_bias_audit,
    'raw_runs': {
        'base': base_res,
        'c1_raw_broadcast': c1_results,
        'c2_raw_date_token': c2_results,
        'c3_raw_sigma0': c3_results,
        'c4_pt_intra_loss': c4_results
    }
}

out_file = 'TICKTACK_CONTROL_MATRIX_RESULTS.json'
with open(out_file, 'w', encoding='utf-8') as f:
    json.dump(final_export, f, indent=2, ensure_ascii=False)
print(f'\n[DONE] Successfully saved full benchmark results to {out_file}!')
